In [1]:
import os
import sys
import numpy as np
import pandas as pd
import netCDF4 as nc
import xgboost
import shap

import dask.array as da
from dask.distributed import Client, LocalCluster
from xgboost.dask import DaskXGBRegressor
from sklearn.model_selection import train_test_split, KFold
from dask import delayed

/home/z5218916/miniconda3/envs/science/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Function to perform the 5-fold cross-validation and out-of-sample test
def train_model(client: Client, X: da.Array, y: da.Array, n_splits=5, test_size=0.2):
    
    # Split the data into training and testing sets (20% test, 80% training)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

    # Set up DaskXGBRegressor
    regressor = DaskXGBRegressor(verbosity=1, n_estimators=100, tree_method="hist", client=client)
    
    # Perform 5-fold cross-validation
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    cv_results = []
    
    for train_index, val_index in kf.split(X_train.compute(), y_train.compute()):
        
        X_train_cv, X_val_cv = X_train[train_index], X_train[val_index]
        y_train_cv, y_val_cv = y_train[train_index], y_train[val_index]
        
        # Train the model
        regressor.fit(X_train_cv, y_train_cv, eval_set=[(X_val_cv, y_val_cv)])
        
        # Make predictions and evaluate the model
        predictions = regressor.predict(X_val_cv)
        # Here you can add evaluation metrics like RMSE, R-squared, etc.
        # For simplicity, we store the predictions and actuals.
        cv_results.append((predictions, y_val_cv))

    # After cross-validation, train on the full training set
    regressor.fit(X_train, y_train, eval_set=[(X_test, y_test)])

    # Evaluate the final model on the test set
    final_predictions = regressor.predict(X_test)
    print("Out-of-sample predictions:", final_predictions)

    return regressor, cv_results

In [4]:
# Start Dask client and cluster
with LocalCluster(n_workers=4, threads_per_worker=1) as cluster:
    with Client(cluster) as client:
        # Load your dataset (replace with your actual file paths and data)
        var_input = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/Qle_all_sites.csv',
                                usecols=['time', 'site_name', 'model_CABLE', 'VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown'],
                                na_values=[''])
        var_input['CABLE_SMtop1m'] = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/SMtop1m_all_sites.csv',
                                                    usecols=['CABLE_SMtop1m'], na_values=[''])
        var_input['obs_LAI'] = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/LAI_all_sites_parallel.csv',
                                            usecols=['obs_LAI'], na_values=[''])

        # select a site for trial 
        site_name = 'AU-Tum'
        var_select = var_input[var_input['site_name'] == site_name]

        # Extract the features and target
        X = var_select[['VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown', 'CABLE_SMtop1m', 'obs_LAI']].values
        y = var_select['model_CABLE'].values

        # Convert to Dask arrays
        X_dask = da.from_array(X, chunks=(1000, X.shape[1]))  # Adjust chunk size as needed
        y_dask = da.from_array(y, chunks=1000)

        # Train the model and get the results
        model, results = train_model(client, X_dask, y_dask)

        # Optionally print the cross-validation results
        print("Cross-validation results:", results)

/home/z5218916/miniconda3/envs/science/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 11.50 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2025-03-04 15:44:13,039 - distributed.protocol.core - CRITICAL - Failed to deserialize
Traceback (most recent call last):
  File "/home/z5218916/miniconda3/envs/science/lib/python3.12/site-packages/distributed/protocol/core.py", line 175, in loads
    return msgpack.loads(
           ^^^^^^^^^^^^^^
  File "/home/z5218916/miniconda3/envs/science/lib/python3.12/site-packages/msgpack/fallback.py", line 128, in unpackb
    ret = unpacker._unpack()
          ^^^^^^^^^^^^^^^^^^
  File "/home/z5218916/miniconda3/envs/science/lib/python3.12/site-packages/msgpack/fallba

FutureCancelledError: ('getitem-b7eec0ceb767455b59c8c88a362f622f', 5, 0) cancelled for reason: scheduler-connection-lost.
Client lost the connection to the scheduler. Please check your connection and re-run your work.

In [ ]:
# var_input = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/Qle_all_sites.csv',
#                         usecols=['time','model_CABLE', 'VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown'],
#                         na_values=[''])

# var_input['CABLE_SMtop1m'] = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/SMtop1m_all_sites.csv',usecols=['CABLE_SMtop1m'],na_values=[''])
# var_input['obs_LAI']       = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/LAI_all_sites_parallel.csv',usecols=['obs_LAI'],na_values=[''])

# model_CABLE                = var_input['model_CABLE'][:]

# var_input.drop(columns=['time','model_CABLE'], inplace=True)

# print(type(var_input))
# print(type(model_CABLE))

# # Used columns in Qle_all_sites.csv 

# # 'model_CABLE-POP-CN',
# # 'model_CHTESSEL_Ref_exp1',
# # 'model_CLM5a',
# # 'model_GFDL',
# # 'model_JULES_GL9',
# # 'model_JULES_GL9_withLAI',
# # 'model_MATSIRO',
# # 'model_MuSICA',
# # 'model_NASAEnt',
# # 'model_NoahMPv401',
# # 'model_ORC2_r6593',
# # 'model_ORC3_r8120',
# # 'model_QUINCY',
# # 'model_STEMMUS-SCOPE',
# # 'obs',
# # 'obs_EF',
# # 'NoahMPv401_greenness',
# # 'month',
# # 'hour',
# # 'site_name',
# # 'IGBP_type',
# # 'climate_type',
# # 'half_hrs_after_precip',
# # 'obs_Qair',